In [ ]:
import sys
sys.path.append('src')s

from config import ExperimentConfig, save_result, load_results
from classical import factor_classical
from shor import factor_shor
from vqf import factor_vqf
from utils import *

import json
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
cfg_sim = ExperimentConfig(
    shots=4096, n_restarts=3,
    maxiter_sim=500, maxiter_hw=150,
    s=2, mode='sim',
    backend_name=None, optimization_level=1
)

cfg_hw = ExperimentConfig(
    shots=4096, n_restarts=3,
    maxiter_sim=500, maxiter_hw=150,
    s=2, mode='hardware',
    backend_name='ibm_brisbane', optimization_level=1
)

N_tier1 = [15, 21, 33, 35, 51, 55, 65, 77, 85, 91]

## Classical

In [ ]:
for N in N_tier1:
    result, _ = factor_classical(N, cfg_sim)
    save_result(result, '../results/classical_results.json')
    print(f"N={N}: {result.factors}")

## Shor

In [ ]:
for N in N_tier1:
    result, circ = factor_shor(N, cfg_sim)
    save_result(result, '../results/shor_results.json')
    print(f"N={N}: {result.factors}, depth={result.circuit_depth}")

## VQF

### Preprocessing

In [ ]:
from vqf_preprocessing import preprocess
from vqf_qubo import energy, reduce_to_quadratic, qubit_list

for N in N_tier1:
    p, q, z, known, sub, clauses = preprocess(N)
    E2 = reduce_to_quadratic(energy(clauses))
    qubits = qubit_list(E2, p, q, known, sub)
    print(f"N={N}: {len(qubits)} qubits")

### Simulation

In [ ]:
for N in N_tier1:
    try:
        result, circ = factor_vqf(N, cfg_sim)
        save_result(result, '../results/vqf_sim_results.json')
        if N == 15: 
            vqf_circ_15 = circ:
        print(f"N={N}: success={result.success}, E_opt={result.E_opt}")
    except MemoryError:
        print(f"N={N}: MemoryError — skipped")
    except Exception as e:
        print(f"N={N}: Failed — {e}")

### QAOA layer sweep for N=15

In [ ]:
for s in [1, 2, 3, 4]:
    cfg_sweep = ExperimentConfig(
        shots=4096, n_restarts=3,
        maxiter_sim=500, s=s, mode='sim'
    )
    result, _ = factor_vqf(15, cfg_sweep)
    save_result(result, '../results/vqf_layer_sweep.json')
    print(f"s={s}: success_prob={result.success_prob:.3f}, E_opt={result.E_opt:.4f}")

### Hardware

In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account(channel='ibm_quantum', token='YOUR_TOKEN', overwrite=True)

In [ ]:
for N in [15, 21]:
    try:
        result, circ = factor_vqf(N, cfg_hw)
        save_result(result, '../results/vqf_hw_results.json')
        print(f"N={N}: success={result.success}, E_opt={result.E_opt}")
    except MemoryError:
        print(f"N={N}: MemoryError — skipped")
    except Exception as e:
        print(f"N={N}: Failed — {e}")

## Results

In [ ]:
classical = load_results('../results/classical_results.json')
shor      = load_results('../results/shor_results.json')
vqf_sim   = load_results('../results/vqf_sim_results.json')
vqf_hw    = load_results('../results/vqf_hw_results.json')
vqf_sweep = load_results('../results/vqf_layer_sweep.json')

In [ ]:
plot_runtime(classical, shor, vqf_sim, vqf_hw)
plot_circuit_metrics(shor, vqf_sim, vqf_hw)
plot_success_prob(vqf_sim, vqf_hw)
plot_success_vs_layers(vqf_sweep)
plot_gate_count(shor, vqf_hw)

## Circuit Diagrams

In [ ]:
# Run shor and vqf for N=15 to get circuits
_, shor_circ = factor_shor(15, cfg_sim)
_, vqf_circ  = factor_vqf(15, cfg_sim)

shor_circ.draw('mpl', fold=40)